# TSA Standalone (stable)

Version autonome du TSA (coarse + fine), allégée pour éviter le crash kernel.

Fichiers attendus:
- `signal.csv`
- `PRN-<satellite>.csv` (ex: `PRN-25.csv`)


In [ ]:
import numpy as np
import time, os

FS = 11.999e6
NB_ECHANTILLONS = 11999
NB_PHASES = 1023
NB_DOPPLER_COARSE = 41
NB_DOPPLER_FINE = 21
FREQUENCE_CENTRALE = 3.563e6
DOPPLER_BIN_COARSE = 500
DOPPLER_BIN_FINE = 250
SEUIL_VARIANCE_COARSE = 0.039
SEUIL_VARIANCE_FINE = 0.18
SHIFTS = np.floor(np.arange(NB_PHASES) * NB_ECHANTILLONS / NB_PHASES).astype(np.int32)

def charger_signal(path):
    return np.asarray(np.loadtxt(path, delimiter=','), dtype=np.float32)

def charger_prn(satellite):
    filename = f"PRN-{satellite}.csv"
    if not os.path.exists(filename):
        raise FileNotFoundError(f"PRN introuvable: {filename}")
    return np.asarray(np.loadtxt(filename, delimiter=','), dtype=np.float32)

def insert_peak(peaks, peak, max_size):
    if len(peaks) < max_size:
        peaks.append(peak)
        return
    i_min = min(range(len(peaks)), key=lambda i: peaks[i][2])
    if peak[2] > peaks[i_min][2]:
        peaks[i_min] = peak

def precompute_prn_shifts(prn, shifts, N, dtype=np.float16):
    out = np.empty((NB_PHASES, N), dtype=dtype)
    for tau in range(NB_PHASES):
        out[tau] = np.roll(prn, -int(shifts[tau]))
    return out

def suivi_gps_fin(signal, doppler_offset):
    print(f"[FINE] start | doppler_offset coarse={doppler_offset}")
    Ts = 1.0 / FS
    n = np.arange(NB_ECHANTILLONS, dtype=np.float32)
    doppler_offsets = np.arange(
        doppler_offset - (NB_DOPPLER_FINE // 2) * DOPPLER_BIN_FINE,
        doppler_offset + (NB_DOPPLER_FINE // 2) * DOPPLER_BIN_FINE + 1,
        DOPPLER_BIN_FINE,
        dtype=np.float32,
    )

    print(f"[FINE] doppler bins={len(doppler_offsets)} | range=[{doppler_offsets[0]}, {doppler_offsets[-1]}]")

    prn_matrices = [PRN_SHIFTED, PRN_SHIFTED_HALF_PLUS, PRN_SHIFTED_HALF_MINUS]
    peaks_by_variant = [[], [], []]
    ratios = [[1.0], [1.0], [1.0]]
    corr_matrix = np.zeros((len(doppler_offsets), NB_PHASES), dtype=np.float32)

    for d, fd in enumerate(doppler_offsets):
        if (d == 0) or (d == len(doppler_offsets) // 2) or (d == len(doppler_offsets) - 1):
            print(f"[FINE] progress doppler {d+1}/{len(doppler_offsets)} | fd={fd}")
        carrier = np.exp(-1j * 2 * np.pi * (FREQUENCE_CENTRALE + fd) * n * Ts).astype(np.complex64)
        dataMix = signal * carrier
        for p, prn_matrix in enumerate(prn_matrices):
            corr_complex = prn_matrix @ dataMix
            power = np.abs(corr_complex).astype(np.float32) ** 2
            if p == 0:
                corr_matrix[d, :] = power
            i = int(np.argmax(power))
            insert_peak(peaks_by_variant[p], (float(fd), i, float(power[i])), 16)

    for p in range(3):
        peaks_by_variant[p].sort(key=lambda x: x[2])
        low = peaks_by_variant[p][0][2]
        high = peaks_by_variant[p][-1][2]
        ratios[p].append(low / high if high > 0 else 0.0)

    var = max(np.var(r) for r in ratios)
    best = None
    for p in peaks_by_variant:
        if p:
            cand = max(p, key=lambda x: x[2])
            if (best is None) or (cand[2] > best[2]):
                best = cand

    detected = 1 if var > SEUIL_VARIANCE_FINE else 0
    return best[0], best[1], best[2], float(var), corr_matrix, detected

def suivi_gps_grossier(signal, precision=8):
    print(f"[COARSE] start | precision={precision} | doppler_bins={NB_DOPPLER_COARSE}")
    doppler_offsets = DOPPLER_BIN_COARSE * (
        np.arange(NB_DOPPLER_COARSE, dtype=np.float32) - (NB_DOPPLER_COARSE - 1) / 2
    )
    ratios = [1.0]

    for p in range(precision):
        print(f"[COARSE] pass {p+1}/{precision}")
        peaks = []
        n = STSA_INDICES[p]
        sig = signal[n]
        prn_sub = PRN_SHIFTED[:, n]
        carriers = np.array([DOPPLER_CARRIERS_COARSE[d][n] for d in range(len(doppler_offsets))], dtype=np.complex64)
        dataMix = carriers * sig
        corr = prn_sub @ dataMix.T
        power = (np.abs(corr).astype(np.float32)) ** 2

        phase_idx = np.argmax(power, axis=0)
        peak_pow = power[phase_idx, np.arange(power.shape[1])]
        for d, fd in enumerate(doppler_offsets):
            insert_peak(peaks, (float(fd), int(phase_idx[d]), float(peak_pow[d])), 5)

        if len(peaks) >= 2:
            peaks.sort(key=lambda x: x[2])
            low, high = peaks[0][2], peaks[-1][2]
            ratios.append(low / high if high > 0 else 0.0)
            if len(ratios) > precision + 1:
                ratios.pop(0)
            var = float(np.var(ratios))
            print(f"[COARSE] variance pass {p+1}: {var:.6f}")
            if var > SEUIL_VARIANCE_COARSE:
                print(f"[COARSE] detecte a la pass {p+1} (seuil={SEUIL_VARIANCE_COARSE})")
                best = max(peaks, key=lambda x: x[2])
                return suivi_gps_fin(signal, best[0])
    print("[COARSE] non detecte")
    return None

def tsa_acquisition(signal):
    out = suivi_gps_grossier(signal)
    if out is None:
        return 0, 0, 0, 0, None, 0
    return out


In [ ]:
# ===== EXECUTION =====
satellite = 25
print(f"[INIT] satellite={satellite}")
print("[INIT] chargement CSV...")
signal = charger_signal("signal.csv")[:NB_ECHANTILLONS]
prn = charger_prn(satellite)[:NB_ECHANTILLONS]
print(f"[INIT] tailles | signal={signal.shape[0]} | prn={prn.shape[0]}")

if signal.shape[0] < NB_ECHANTILLONS or prn.shape[0] < NB_ECHANTILLONS:
    raise ValueError("signal/prn trop courts")

signal = signal.astype(np.complex64)
prn = prn.astype(np.float32)

print("[INIT] precompute PRN shifts...")
PRN_SHIFTED = precompute_prn_shifts(prn, SHIFTS, NB_ECHANTILLONS, dtype=np.float16)
prn_half_plus = 0.5 * (prn + np.roll(prn, -1))
prn_half_minus = 0.5 * (prn + np.roll(prn, 1))
PRN_SHIFTED_HALF_PLUS = precompute_prn_shifts(prn_half_plus, SHIFTS, NB_ECHANTILLONS, dtype=np.float16)
PRN_SHIFTED_HALF_MINUS = precompute_prn_shifts(prn_half_minus, SHIFTS, NB_ECHANTILLONS, dtype=np.float16)

print("[INIT] precompute STSA indices + Doppler carriers coarse...")
STSA_INDICES = [np.arange(i, NB_ECHANTILLONS, 8, dtype=np.int32) for i in range(8)]
n_full = np.arange(NB_ECHANTILLONS, dtype=np.float32)
Ts = 1.0 / FS
doppler_offsets_coarse = DOPPLER_BIN_COARSE * (
    np.arange(NB_DOPPLER_COARSE, dtype=np.float32) - (NB_DOPPLER_COARSE - 1) / 2
)
DOPPLER_CARRIERS_COARSE = [
    np.exp(-1j * 2 * np.pi * (FREQUENCE_CENTRALE + fd) * n_full * Ts).astype(np.complex64)
    for fd in doppler_offsets_coarse
]

print("[RUN] lancement TSA (coarse + fine)...")
t0 = time.time()
doppler, phase, peak, variance, corr_matrix, detected = tsa_acquisition(signal)
dt = time.time() - t0

print("[DONE] resultat final")
print("Detected:", detected)
print("Doppler:", doppler)
print("Phase:", phase)
print("Peak:", peak)
print("Variance:", variance)
print(f"Temps execution: {dt:.4f} s")
